In [38]:
!pip install transformers datasets torchmetrics pytorch-lightning tqdm matplotlib pandas

In [39]:
from transformers import AutoTokenizer, AutoModelForCausalLM, AdamW, get_scheduler
from datasets import load_dataset
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence
from torchmetrics.text.rouge import ROUGEScore
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import torch
from torch import nn
import numpy as np
import random
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import pandas as pd

In [40]:
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

device = torch.device("cpu")
print("Using device:", device)

Using device: cuda


In [44]:
from datasets import DatasetDict

# Load datasets and metadata
raw_datasets = DatasetDict({
    "train": load_dataset("json", data_files={"train": "/content/train-open.json"}, split="train[:30%]"),
    "validation": load_dataset("json", data_files="/content/val-open.json", split="train[:15%]"),
    "test": load_dataset("json", data_files="/content/test-open.json", split="train[:15%]"),
})

metadata_path = "/content/all_data_meta.csv"
try:
    metadata = pd.read_csv(metadata_path, on_bad_lines='skip')
    print("Metadata cleaned and loaded successfully. Structure:")
    print(metadata.head())
except Exception as e:
    print("Error loading metadata:", e)

# Clean metadata by dropping rows with missing or invalid values
def clean_metadata(metadata_df):
    metadata_df = metadata_df.dropna()  # Remove rows with missing values
    return metadata_df

metadata = clean_metadata(metadata)

Metadata cleaned and loaded successfully. Structure:
   answer_id  question_id                                               text  \
0     649960       649960  المعلومات المخزنة أو وسيلة التخزين.\nأي أن الب...   
1     649961       649961   يتكون البايت عادة من 8 بت، ولذلك فأن البايت ي...   
2     649962       649962  لاحظ أن الأسماء كيلوبايت وميجابايت ...الخ، يمك...   
3     649963       649963  وبتمديد النمط، نستطيع الحصول على وحدتين إضافيت...   
4     649959       649959  البايت أو الثُّمَانِيَّة هي وحدة معلومات رقمية...   

   answer_start  answer_end  answer_category                  question  \
0           251         418              NaN      اذكر وسيله التخزين ؟   
1           417         536              NaN       مما يتكون البايت  ؟   
2           748         874              NaN    ما هي مضاعفات البايت ؟   
3           882        1015              NaN  ما هو الاستخدام البايت ؟   
4             0         233              NaN  ما هو البايت ف الحاسوب ؟   

   file_name open-dom

In [45]:
MODEL_NAME = "aubmindlab/aragpt2-base"

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token  # Set pad_token to eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)

In [47]:
def preprocess_function(examples):
    # Align metadata fields if necessary
    meta_dict = metadata.set_index("question_id").to_dict("index")
    inputs = []
    for q_id, question, answer in zip(examples["question_id"], examples["question"], examples["answer"]):
        if q_id in meta_dict:
            meta = meta_dict[q_id]
            context = f"Category: {meta['answer_category']} | File: {meta['file_name']}"
        else:
            context = ""
        inputs.append(f"question: {question} context: {context}")

    targets = examples["answer"]
    inputs = tokenizer(inputs, truncation=True, padding="max_length", max_length=256)
    targets = tokenizer(targets, truncation=True, padding="max_length", max_length=256)

    inputs["labels"] = targets["input_ids"]  # Ensure labels align with input_ids
    inputs["attention_mask"] = inputs["attention_mask"]
    return inputs

tokenized_datasets = raw_datasets.map(preprocess_function, batched=True)

def preprocess_function(examples):
    inputs = ["question: " + q for q in examples["question"]]
    targets = examples["answer"]
    inputs = tokenizer(inputs, truncation=True, padding="max_length", max_length=128)
    targets = tokenizer(targets, truncation=True, padding="max_length", max_length=128)

    inputs["labels"] = targets["input_ids"]  # Ensure labels align with input_ids
    inputs["attention_mask"] = inputs["attention_mask"]
    return inputs

tokenized_datasets = raw_datasets.map(preprocess_function, batched=True)

Map:   0%|          | 0/17603 [00:00<?, ? examples/s]

Map:   0%|          | 0/1907 [00:00<?, ? examples/s]

Map:   0%|          | 0/1889 [00:00<?, ? examples/s]

Map:   0%|          | 0/17603 [00:00<?, ? examples/s]

Map:   0%|          | 0/1907 [00:00<?, ? examples/s]

Map:   0%|          | 0/1889 [00:00<?, ? examples/s]

In [48]:
def collate_fn(batch):
    input_ids = pad_sequence([torch.tensor(x["input_ids"]) for x in batch], batch_first=True, padding_value=tokenizer.pad_token_id)
    attention_mask = pad_sequence([torch.tensor(x["attention_mask"]) for x in batch], batch_first=True, padding_value=0)
    labels = pad_sequence([torch.tensor(x["labels"]) for x in batch], batch_first=True, padding_value=-100)
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

train_dataloader = DataLoader(tokenized_datasets["train"], batch_size=8, shuffle=True, collate_fn=collate_fn)
val_dataloader = DataLoader(tokenized_datasets["validation"], batch_size=8, collate_fn=collate_fn)
test_dataloader = DataLoader(tokenized_datasets["test"], batch_size=8, collate_fn=collate_fn)

### Cell 8: Define Optimizer and Scheduler
optimizer = AdamW(model.parameters(), lr=1e-5)
scheduler = get_scheduler(
    "cosine", optimizer=optimizer, num_warmup_steps=50, num_training_steps=len(train_dataloader) * 10
)

/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


In [49]:
model.config.dropout = 0.3

def train_epoch(model, dataloader, optimizer, scheduler, history):
    model.train()
    total_loss = 0
    progress_bar = tqdm(dataloader, desc="Training", leave=False)
    for batch in progress_bar:
        optimizer.zero_grad()
        inputs = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        outputs = model(input_ids=inputs, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        progress_bar.set_postfix(loss=loss.item())
    avg_loss = total_loss / len(dataloader)
    history['train_loss'].append(avg_loss)
    return avg_loss

In [50]:
def evaluate_epoch(model, dataloader, history):
    model.eval()
    total_loss = 0
    progress_bar = tqdm(dataloader, desc="Validation", leave=False)
    with torch.no_grad():
        for batch in progress_bar:
            inputs = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            outputs = model(input_ids=inputs, attention_mask=attention_mask, labels=labels)
            total_loss += outputs.loss.item()
            progress_bar.set_postfix(loss=outputs.loss.item())
    avg_loss = total_loss / len(dataloader)
    history['val_loss'].append(avg_loss)
    return avg_loss

In [ ]:
history = {'train_loss': [], 'val_loss': []}
EPOCHS = 20
for epoch in range(EPOCHS):
    print(f"Epoch {epoch + 1}/{EPOCHS}")
    train_loss = train_epoch(model, train_dataloader, optimizer, scheduler, history)
    val_loss = evaluate_epoch(model, val_dataloader, history)
    print(f"Train Loss: {train_loss:.4f}, Validation Loss: {val_loss:.4f}")

### Cell 12: Save Fine-Tuned Model
model.save_pretrained("./fine_tuned_gpt2")
tokenizer.save_pretrained("./fine_tuned_gpt2")

Epoch 1/20


Training:   0%|          | 0/2201 [00:00<?, ?it/s]

Validation:   0%|          | 0/239 [00:00<?, ?it/s]

Train Loss: 1.1799, Validation Loss: 1.0133
Epoch 2/20


Training:   0%|          | 0/2201 [00:00<?, ?it/s]

In [27]:
EPOCHS = 5
for epoch in range(EPOCHS):
    print(f"Epoch {epoch + 1}/{EPOCHS}")
    train_loss = train_epoch(model, train_dataloader, optimizer, scheduler)
    val_loss = evaluate_epoch(model, val_dataloader)
    print(f"Train Loss: {train_loss:.4f}, Validation Loss: {val_loss:.4f}")

Epoch 1/5


Training:   0%|          | 0/734 [00:00<?, ?it/s]

Validation:   0%|          | 0/80 [00:00<?, ?it/s]

Train Loss: 1.4550, Validation Loss: 1.1360
Epoch 2/5


Training:   0%|          | 0/734 [00:00<?, ?it/s]

...

Validation:   0%|          | 0/80 [00:00<?, ?it/s]

Train Loss: 0.9998, Validation Loss: 1.0460
Epoch 5/5


Training:   0%|          | 0/734 [00:00<?, ?it/s]

Validation:   0%|          | 0/80 [00:00<?, ?it/s]

Train Loss: 0.9918, Validation Loss: 1.0479


In [28]:
model.save_pretrained("./fine_tuned_gpt2")
tokenizer.save_pretrained("./fine_tuned_gpt2")

('./fine_tuned_gpt2/tokenizer_config.json',
 './fine_tuned_gpt2/special_tokens_map.json',
 './fine_tuned_gpt2/vocab.json',
 './fine_tuned_gpt2/merges.txt',
 './fine_tuned_gpt2/added_tokens.json',
 './fine_tuned_gpt2/tokenizer.json')

In [29]:
def test_model(question):
    inputs = tokenizer("question: " + question, return_tensors="pt", padding=True).to(device)
    outputs = model.generate(inputs["input_ids"], max_length=50)
    print("Answer:", tokenizer.decode(outputs[0], skip_special_tokens=True))

test_question = "What is the capital of Morocco?"
test_model(test_question)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


Answer: question: What is the capital of Morocco?


In [30]:
def evaluate_model_on_test_set():
    model.eval()
    y_true, y_pred = [], []
    progress_bar = tqdm(test_dataloader, desc="Testing", leave=False)
    with torch.no_grad():
        for batch in progress_bar:
            inputs = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model.generate(inputs, attention_mask=attention_mask, max_length=50)
            predictions = [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]

            for pred, label in zip(predictions, labels):
                y_pred.append(pred)
                y_true.append(tokenizer.decode(label[label != -100], skip_special_tokens=True))

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average="weighted")
    recall = recall_score(y_true, y_pred, average="weighted")
    f1 = f1_score(y_true, y_pred, average="weighted")

    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")

evaluate_model_on_test_set()

NameError: name 'test_dataloader' is not defined